In [ ]:
# View dimensional coverage for a specific trace
if len(df_coverage) > 0:
    # Pick first trace as example
    trace_row = df_coverage.iloc[0]
    coverage = trace_row['dimensionalCoverage']
    
    print(f"\n{'='*60}")
    print(f"TRACE: {trace_row['trace_id']}")
    print(f"{'='*60}")
    print(f"Timestamp: {trace_row['timestamp']}")
    print(f"Variant: {trace_row['experimentVariant']}")
    print(f"\nCoverage: {coverage['summary']['dimensionsCovered']}/10 dimensions ({coverage['summary']['coveragePercentage']}%)")
    print(f"Primary (high confidence): {len(coverage['summary']['primaryDimensions'])}")
    print(f"Gaps: {len(coverage['summary']['gaps'])}")
    
    print(f"\n{'='*60}")
    print("DIMENSION BREAKDOWN")
    print(f"{'='*60}\n")
    
    for dimension, data in coverage['dimensions'].items():
        status = "✅" if data['covered'] else "❌"
        conf = data['confidence'].upper() if data['covered'] else "N/A"
        themes = ", ".join(data['themes']) if data['themes'] else "None"
        
        print(f"{status} {dimension}")
        if data['covered']:
            print(f"   Confidence: {conf}")
            print(f"   Themes: {themes}")
        print()
    
    if coverage['summary']['gaps']:
        print(f"\n⚠️  Missing dimensions: {', '.join(coverage['summary']['gaps'])}")

In [ ]:
# Identify systematic gaps (dimensions covered in <50% of conversations)
if len(df_coverage) > 0:
    gaps = find_systematic_gaps(df_coverage, threshold=0.5)
    
    print(f"\n{'='*60}")
    print("SYSTEMATIC GAPS (Coverage < 50%)")
    print(f"{'='*60}\n")
    
    if len(gaps) > 0:
        print(f"Found {len(gaps)} dimensions that are systematically under-covered:\n")
        print(gaps.to_string(index=False))
        
        print("\n⚠️  These dimensions may need proactive questioning in future experiments")
    else:
        print("✅ No systematic gaps! All dimensions covered in >50% of conversations")

In [ ]:
# Coverage patterns: Which dimensions are typically covered?
if len(df_coverage) > 0:
    coverage_patterns = analyze_coverage_patterns(df_coverage)
    
    print(f"\n{'='*60}")
    print("DIMENSION COVERAGE PATTERNS")
    print(f"{'='*60}\n")
    print(coverage_patterns.to_string(index=False))
    
    # Visualize if matplotlib is available
    try:
        import matplotlib.pyplot as plt
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Coverage rate
        ax1.barh(coverage_patterns['dimension'], coverage_patterns['coverage_rate'])
        ax1.set_xlabel('Coverage Rate')
        ax1.set_title('Dimension Coverage Rate')
        ax1.set_xlim(0, 1)
        
        # Average confidence when covered
        covered_dims = coverage_patterns[coverage_patterns['coverage_rate'] > 0]
        ax2.barh(covered_dims['dimension'], covered_dims['avg_confidence_when_covered'])
        ax2.set_xlabel('Avg Confidence (1=low, 2=medium, 3=high)')
        ax2.set_title('Average Confidence When Covered')
        ax2.set_xlim(0, 3)
        
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("\n(Install matplotlib for visualizations: pip install matplotlib)")

In [ ]:
# Cell: Dimensional Coverage Analysis (Experiment 2)
# After backfilling with: npx tsx scripts/backfill-dimensional-coverage.ts

import sys
sys.path.append('..')
from scripts.dimensional_coverage_analysis import (
    load_dimensional_coverage,
    analyze_coverage_patterns,
    find_systematic_gaps,
    get_coverage_summary_stats,
    get_theme_to_dimension_mapping
)

# Load traces with dimensional coverage
df_coverage = load_dimensional_coverage()
print(f"Loaded {len(df_coverage)} traces with dimensional coverage")

if len(df_coverage) > 0:
    # Overall summary statistics
    summary = get_coverage_summary_stats(df_coverage)
    print(f"\n{'='*60}")
    print("OVERALL DIMENSIONAL COVERAGE SUMMARY")
    print(f"{'='*60}")
    print(f"Total traces analyzed: {summary['total_traces']}")
    print(f"Avg dimensions covered: {summary['avg_dimensions_covered']:.1f} / 10")
    print(f"Avg coverage percentage: {summary['avg_coverage_percentage']:.1f}%")
    print(f"Avg primary (high confidence) dimensions: {summary['avg_primary_dimensions']:.1f}")
else:
    print("\nNo traces with dimensional coverage yet")
    print("Run: npx tsx scripts/backfill-dimensional-coverage.ts")

# Trace Analysis Starter

This notebook demonstrates the basic workflow for trace analysis and error coding.

## Setup

Make sure you've:
1. Created a virtual environment: `python3 -m venv venv`
2. Activated it: `source venv/bin/activate`
3. Installed dependencies: `pip install -r requirements.txt`
4. Have DATABASE_URL in your `.env.local`

In [ ]:
# Cell 1: Setup
import sys
sys.path.append('..')  # Add parent directory to path

from scripts.trace_analysis import TraceAnalyzer
import pandas as pd

# Connect to database
analyzer = TraceAnalyzer()

In [ ]:
# Cell 2: Load traces
traces = analyzer.load_traces(limit=50)
print(f"Loaded {len(traces)} traces")

# Quick overview
traces[['id', 'selectedLens', 'questionCount', 'userFeedback', 'reviewedAt']].head()

In [ ]:
# Cell 3: Filter traces (example)
# Example: Find all lens A conversations with negative feedback
problematic = traces[
    (traces['selectedLens'] == 'A') &
    (traces['userFeedback'] == 'not_helpful')
]
print(f"Found {len(problematic)} problematic traces")
problematic[['id', 'questionCount', 'timestamp']]

In [ ]:
# Cell 4: Review individual trace with events
# Pick first trace from filtered set (or use any trace ID)
if len(traces) > 0:
    trace_id = traces.iloc[0]['id']
    analyzer.display_trace(trace_id)
    
    # Load events for this trace
    events = analyzer.load_events(trace_id=trace_id)
    if len(events) > 0:
        print(f"\n{'='*60}")
        print(f"EVENTS FOR THIS TRACE ({len(events)} events)")
        print(f"{'='*60}\n")
        for _, event in events.iterrows():
            print(f"[{event['timestamp']}] {event['eventType']}")
            print(f"  Data: {event['eventData']}")
            print()
    else:
        print("\nNo events recorded for this trace")
else:
    print("No traces available. Run the app and create some conversations first.")

In [ ]:
# Cell 5: Add open coding notes
# Record observations as you review
# analyzer.annotate_trace(
#     trace_id=trace_id,
#     notes="""
#     Observation: User provided specific customer pain point,
#     but extraction generalized too much.
#     
#     Potential error: over-generalization in extraction phase
#     Impact: Strategy output was too vague
#     """
# )

In [ ]:
# Cell 6: View all coded traces
coded = analyzer.load_traces(limit=100, filters={'hasNotes': True})
print(f"{len(coded)} traces have been coded")

# Browse notes
for idx, row in coded.iterrows():
    print(f"\n--- Trace {row['id'][:8]}... ---")
    print(f"Lens: {row['selectedLens']}, Feedback: {row['userFeedback']}")
    notes = row['openCodingNotes']
    if notes:
        print(notes[:200] + "..." if len(notes) > 200 else notes)

In [ ]:
# Cell 7: Apply categories (after synthesis)
# Once you've identified themes, apply categories
# Example: Categorize traces with extraction issues

# analyzer.categorize_trace(
#     trace_id=trace_id,
#     categories=['extraction-overgeneralization', 'weak-strategy-specificity']
# )

In [ ]:
# Cell 8: Summary statistics
summary = analyzer.get_coding_summary()

print(f"Total traces: {summary['total_traces']}")
print(f"Coded: {summary['coded_traces']} ({summary['coded_traces']/summary['total_traces']*100:.1f}%)")
print(f"Categorized: {summary['categorized_traces']}")
print("\nCategory distribution:")
print(summary['category_distribution'])

## Event Analysis

Analyze user interaction events (fake door clicks, info icon views, extraction choices)

In [ ]:
# Cell 9: Load all events
all_events = analyzer.load_events(limit=1000)
print(f"Total events recorded: {len(all_events)}")
print(f"\nEvent type distribution:")
print(all_events['eventType'].value_counts())
print(f"\nRecent events:")
all_events.head(10)

In [ ]:
# Cell 10: Analyze fake door clicks (feature demand)
fake_door_clicks = all_events[all_events['eventType'] == 'fake_door_click']

if len(fake_door_clicks) > 0:
    print(f"Total fake door clicks: {len(fake_door_clicks)}")
    print("\nMost clicked features:")
    
    # Extract feature from eventData
    features = fake_door_clicks['eventData'].apply(lambda x: x.get('feature', 'Unknown'))
    feature_counts = features.value_counts()
    
    for feature, count in feature_counts.items():
        print(f"  {feature}: {count} clicks")
else:
    print("No fake door clicks yet")

In [ ]:
# Cell 14: Compare experiment variants
# Compare quality and engagement across different variants

if len(traces) > 0:
    # Get traces with experiment variants
    variant_traces = traces[traces['experimentVariant'].notna()]
    
    if len(variant_traces) > 0:
        variants = variant_traces['experimentVariant'].unique()
        print(f"Experiment variants found: {len(variants)}")
        
        for variant in variants:
            variant_data = variant_traces[variant_traces['experimentVariant'] == variant]
            print(f"\n{'='*60}")
            print(f"{variant.upper()} (n={len(variant_data)})")
            print(f"{'='*60}")
            
            # Quality rating distribution
            if variant_data['qualityRating'].notna().any():
                print("\nQuality ratings:")
                quality_dist = variant_data['qualityRating'].value_counts()
                for rating, count in quality_dist.items():
                    pct = (count / len(variant_data[variant_data['qualityRating'].notna()])) * 100
                    print(f"  {rating}: {count} ({pct:.1f}%)")
            else:
                print("\nNo quality ratings yet")
            
            # Event engagement
            variant_trace_ids = variant_data['id'].tolist()
            variant_events = all_events[all_events['traceId'].isin(variant_trace_ids)]
            
            if len(variant_events) > 0:
                print(f"\nEvent engagement:")
                print(f"  Total events: {len(variant_events)}")
                print(f"  Avg per trace: {len(variant_events) / len(variant_data):.1f}")
                print(f"\n  Breakdown:")
                for event_type, count in variant_events['eventType'].value_counts().items():
                    avg = count / len(variant_data)
                    print(f"    {event_type}: {avg:.2f} per trace")
    else:
        print("No experiment variants tagged yet")
else:
    print("No traces yet")

In [ ]:
# Cell 13: Correlate events with quality ratings
# Compare engagement patterns between good vs bad rated strategies

if len(traces) > 0:
    # Get traces with quality ratings
    rated_traces = traces[traces['qualityRating'].notna()]
    
    if len(rated_traces) > 0:
        print(f"Traces with quality ratings: {len(rated_traces)}")
        print(f"\nRating distribution:")
        print(rated_traces['qualityRating'].value_counts())
        
        # Compare event counts by rating
        for rating in ['good', 'bad']:
            rating_traces = rated_traces[rated_traces['qualityRating'] == rating]
            if len(rating_traces) > 0:
                trace_ids = rating_traces['id'].tolist()
                rating_events = all_events[all_events['traceId'].isin(trace_ids)]
                
                print(f"\n{rating.upper()} rated traces ({len(rating_traces)} traces):")
                print(f"  Average events per trace: {len(rating_events) / len(rating_traces):.1f}")
                
                # Breakdown by event type
                event_breakdown = rating_events['eventType'].value_counts()
                for event_type, count in event_breakdown.items():
                    avg = count / len(rating_traces)
                    print(f"  - {event_type}: {avg:.1f} per trace")
    else:
        print("No quality ratings yet")
else:
    print("No traces yet")

In [ ]:
# Cell 12: Analyze extraction choices (continue/flag/dismiss)
extraction_choices = all_events[all_events['eventType'] == 'extraction_choice']

if len(extraction_choices) > 0:
    print(f"Total extraction choices: {len(extraction_choices)}")
    print("\nChoice distribution:")
    
    # Extract choice from eventData
    choices = extraction_choices['eventData'].apply(lambda x: x.get('choice', 'Unknown'))
    choice_counts = choices.value_counts()
    
    for choice, count in choice_counts.items():
        pct = (count / len(extraction_choices)) * 100
        print(f"  {choice}: {count} ({pct:.1f}%)")
else:
    print("No extraction choices yet")

In [ ]:
# Cell 11: Analyze info icon views (education engagement)
info_views = all_events[all_events['eventType'] == 'info_icon_view']

if len(info_views) > 0:
    print(f"Total info icon views: {len(info_views)}")
    print("\nMost viewed elements:")
    
    # Extract element from eventData
    elements = info_views['eventData'].apply(lambda x: x.get('element', 'Unknown'))
    element_counts = elements.value_counts()
    
    for element, count in element_counts.items():
        print(f"  {element}: {count} views")
else:
    print("No info icon views yet")